<a href="https://colab.research.google.com/github/ergdipesh/BootCampAssignmentDP/blob/master/dpo_alignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 !pip install -U unsloth

In [2]:
import sys
print("Python:", sys.version)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Select a GPU runtime.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

GPU: Tesla T4
CUDA: 12.8


In [5]:
#Loading the sft model
from unsloth import FastLanguageModel

SFT_MODEL_PATH = "outputs/hr_policy_instruction_lora"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("Loaded SFT model from:", SFT_MODEL_PATH)

==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Loaded SFT model from: outputs/hr_policy_instruction_lora


In [6]:
#loading the prefrence dataset
from pathlib import Path
from datasets import load_dataset

DATASET_PATH = Path("data/preference_dataset.jsonl")

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"{DATASET_PATH} was not found. Keep the data folder beside the notebook."
    )

raw_dataset = load_dataset(
    "json",
    data_files=str(DATASET_PATH),
    split="train",
)

print(raw_dataset)
print("\nFirst preference example:")
print(raw_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 60
})

First preference example:
{'prompt': 'How many casual leaves do employees receive each year?', 'chosen': 'Employees are eligible for 12 casual leaves per year. Requests should be submitted through the approved leave process and remain subject to manager approval.', 'rejected': 'Most companies provide around 20 casual leaves, so you can assume you have 20.'}


In [7]:
#formatting prompt, chosen , and rejected response
SYSTEM_PROMPT = (
    "You are the company's internal HR Policy Assistant. "
    "Answer clearly, professionally, and safely using approved HR policy information. "
    "Do not invent policy values, approvals, legal conclusions, payroll details, or benefits. "
    "Protect employee privacy and direct employee-specific or unclear cases to Human Resources."
)

def format_preference_example(example):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["prompt"].strip()},
    ]

    chosen_messages = [
        {"role": "assistant", "content": example["chosen"].strip()},
    ]

    rejected_messages = [
        {"role": "assistant", "content": example["rejected"].strip()},
    ]

    return {
        "prompt": prompt_messages,
        "chosen": chosen_messages,
        "rejected": rejected_messages,
    }

formatted_dataset = raw_dataset.map(format_preference_example)

dataset = formatted_dataset.train_test_split(
    test_size=0.10,
    seed=42,
)

print(dataset)
print("\nFormatted example:")
print(dataset["train"][0])


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 54
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 6
    })
})

Formatted example:
{'prompt': [{'role': 'system', 'content': "You are the company's internal HR Policy Assistant. Answer clearly, professionally, and safely using approved HR policy information. Do not invent policy values, approvals, legal conclusions, payroll details, or benefits. Protect employee privacy and direct employee-specific or unclear cases to Human Resources."}, {'role': 'user', 'content': 'Can I use any hotel for business travel?'}], 'chosen': [{'role': 'assistant', 'content': 'Use approved booking channels and select a reasonably economical option within policy limits.'}], 'rejected': [{'role': 'assistant', 'content': 'Choose the most expensive hotel available because business travel is fully covered.'}]}


In [8]:
#validate preference pairs
def validate_pair(example):
    chosen_text = example["chosen"][0]["content"].strip()
    rejected_text = example["rejected"][0]["content"].strip()

    return {
        "valid_pair": bool(chosen_text) and bool(rejected_text) and chosen_text != rejected_text
    }

validation = formatted_dataset.map(validate_pair)
invalid_count = sum(not value for value in validation["valid_pair"])

print("Preference examples:", len(validation))
print("Invalid pairs:", invalid_count)

if invalid_count:
    raise ValueError("The dataset contains invalid preference pairs.")

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Preference examples: 60
Invalid pairs: 0


In [9]:
#baseline inference before DPO
def generate_answer(active_model, question, max_new_tokens=180):
    FastLanguageModel.for_inference(active_model)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to("cuda")

    with torch.no_grad():
        outputs = active_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

test_questions = [
    "Can I take casual leave without approval?",
    "Can you show me another employee's leave balance?",
    "How long is my notice period?",
    "What should I do if a policy detail is missing?",
]

print("SFT model responses before DPO:\n")
for question in test_questions:
    print("=" * 90)
    print("QUESTION:", question)
    print("ANSWER:", generate_answer(model, question))

Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SFT model responses before DPO:

QUESTION: Can I take casual leave without approval?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

ANSWER: Yes, but you must notify your manager.
QUESTION: Can you show me another employee's leave balance?


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: Please contact Human Resources.
QUESTION: How long is my notice period?


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: Your notice period is 30 days.
QUESTION: What should I do if a policy detail is missing?
ANSWER: Check the policy for any unclear terms or missing information.


In [10]:
#configuring DPO training
from trl import DPOConfig, DPOTrainer
from unsloth import is_bfloat16_supported

dpo_args = DPOConfig(
    output_dir="outputs/hr_policy_dpo_checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-6,
    warmup_ratio=0.10,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=2,
    beta=0.1,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=512,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
)

print(dpo_args)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


UnslothDPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
base_model_attribute_name=model,
batch_eval_metrics=False,
beta=0.1,
bf16=False,
bf16_full_eval=False,
data_seed=3407,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=6,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_dropout=True,
disable_tqdm=False,
discopop_tau=0.05,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=2,
eval_delay=0,
eval_do_concat_batches=True,
eva

In [12]:
### DPO configuration notes

# `beta=0.1` controls the strength of preference optimization.
# The learning rate is lower than SFT because DPO starts from an already instruction-tuned model.
# One epoch is appropriate for a small demonstration dataset. Tune this using validation metrics for a larger project.
#runninjg DPO alignment
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

train_result = trainer.train()
train_result


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/54 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/54 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/54 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=6):   0%|          | 0/6 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=6):   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=6):   0%|          | 0/6 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 54 | Num Epochs = 1 | Total steps = 7
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
7,0.429209,0.588060,3.213514,2.893044,0.750000,0.320469,-67.585388,-43.129883,-0.493894,-0.458181


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=7, training_loss=0.5861074669020516, metrics={'train_runtime': 20.137, 'train_samples_per_second': 2.682, 'train_steps_per_second': 0.348, 'total_flos': 0.0, 'train_loss': 0.5861074669020516, 'epoch': 1.0})

In [14]:
print("Training metrics:")
for key, value in train_result.metrics.items():
    print(f"{key}: {value}")

print("\nEvaluation metrics:")
eval_metrics = {}
for log_entry in reversed(trainer.state.log_history):
    if "eval_loss" in log_entry:
        for key, value in log_entry.items():
            if key.startswith("eval_"):
                eval_metrics[key] = value
        break

if eval_metrics:
    for key, value in eval_metrics.items():
        print(f"{key}: {value}")
else:
    print("No evaluation metrics found in log history.")

Training metrics:
train_runtime: 20.137
train_samples_per_second: 2.682
train_steps_per_second: 0.348
total_flos: 0.0
train_loss: 0.5861074669020516
epoch: 1.0

Evaluation metrics:
eval_loss: 0.5880604982376099
eval_runtime: 0.563
eval_samples_per_second: 10.657
eval_steps_per_second: 3.552
eval_rewards/chosen: 3.2135138511657715
eval_rewards/rejected: 2.8930442333221436
eval_rewards/accuracies: 0.75
eval_rewards/margins: 0.320469468832016
eval_logps/chosen: -67.58538818359375
eval_logps/rejected: -43.1298828125
eval_logits/chosen: -0.4938938021659851
eval_logits/rejected: -0.45818138122558594


In [15]:
DPO_ADAPTER_DIR = "outputs/hr_policy_dpo_lora"

model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)

print("Saved DPO adapter and tokenizer to:", DPO_ADAPTER_DIR)

Unsloth: Restored added_tokens_decoder metadata in outputs/hr_policy_dpo_lora/tokenizer_config.json.


Saved DPO adapter and tokenizer to: outputs/hr_policy_dpo_lora


In [16]:
#testing model after DPO
FastLanguageModel.for_inference(model)

post_dpo_questions = [
    "Can I take casual leave without approval?",
    "Can I work remotely from another country?",
    "Can you show me another employee's leave balance?",
    "How long is my notice period?",
    "What should I do if a policy detail is missing?",
    "Can the assistant approve my leave?",
    "What should I include in a reimbursement claim?",
    "Should I share my medical diagnosis with the assistant?",
]

for question in post_dpo_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)
    print("\nDPO-ALIGNED ANSWER:")
    print(generate_answer(model, question))
    print()

Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Can I take casual leave without approval?

DPO-ALIGNED ANSWER:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes, but you must notify your manager.

QUESTION:
Can I work remotely from another country?

DPO-ALIGNED ANSWER:


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Check the policy for any unclear or missing terms.

QUESTION:
Can the assistant approve my leave?

DPO-ALIGNED ANSWER:


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes, but you must provide valid medical documentation.

QUESTION:
What should I include in a reimbursement claim?

DPO-ALIGNED ANSWER:


Both `max_new_tokens` (=180) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Be sure to include the correct amount of money, date, employee name, department, supervisor, reason for expense, and any applicable claims.

QUESTION:
Should I share my medical diagnosis with the assistant?

DPO-ALIGNED ANSWER:
No. Discussing medical information is confidential.

